# Exercise: Implementing a GPT Training Loop

Welcome! In this exercise, you will implement the fundamental components of a deep learning training loop from scratch. While the lectures have covered the theory, getting your hands dirty with code is the best way to solidify your understanding. We will be working with a simplified GPT model and a dummy dataset, allowing you to focus purely on the training mechanics.

By the end of this exercise, you will be able to:
*   Implement a function to generate random batches of data for training.
*   Write the core training step, which includes the forward pass, loss calculation, backward pass, and optimizer update.
*   Combine these components into a complete training loop that iteratively improves the model.

Let's get started!

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import copy

# -- For reproducibility --
torch.manual_seed(1337)

# -- Hyperparameters --
# These are simplified for the exercise
batch_size = 16
block_size = 32 # The context length for predictions
vocab_size = 65 # A small vocabulary size for our dummy model
n_embd = 64     # Embedding dimension
learning_rate = 1e-3
# We will run the training for a small number of steps
# to keep things fast for this exercise.
TRAINING_STEPS = 100

# -- Dummy Data and Model --
# We will create a tensor of random integers to serve as our training data.
# Each integer represents a "token" in our vocabulary.
train_data = torch.randint(0, vocab_size, (20000,))

# Here is a simplified GPT model. You don't need to modify this.
# Its `forward` method is the key part: it takes input indices and targets,
# and returns the model's predictions (logits) and the loss.
class DummyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B, T, C) = (Batch, Time, Channels)
        logits = self.lm_head(tok_emb) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

print("Setup complete. Let's begin the exercises!")

### Exercise 1: Creating Data Batches

Our first step is to get data from our dataset into a format the model can use. We don't want to feed the entire dataset to the model at once; instead, we'll take small, random chunks, or "batches".

Your task is to implement the `get_batch` function. This function should:
1.  Randomly select `batch_size` starting positions from the `train_data`.
2.  For each starting position, extract a sequence of length `block_size`. This will be our input `x`.
3.  For each input sequence `x`, create the corresponding target sequence `y`. The target for each token in `x` is the very next token in the original data. So, `y` is just `x` shifted one position to the right.

**Hint:** Use `torch.randint` to generate random starting indices. Then, you can use these indices to slice the `train_data` tensor. `torch.stack` will be helpful for creating the final batch tensors.

In [ ]:
# GRADED FUNCTION: get_batch

def get_batch(data: torch.Tensor, block_size: int, batch_size: int) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Generate a random batch of data of size (batch_size, block_size).

    Args:
        data (torch.Tensor): The full dataset (a 1D tensor of token IDs).
        block_size (int): The context length.
        batch_size (int): The number of sequences in the batch.

    Returns:
        tuple[torch.Tensor, torch.Tensor]: A tuple containing the input batch (x) and the target batch (y).
    """
        block_size (int): The context length.
        batch_size (int): The number of sequences in the batch.

    Returns:
        tuple[torch.Tensor, torch.Tensor]: A tuple containing the input batch (x) and the target batch (y).
    """
    # Generate random starting indices for each sequence in the batch
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # Create the input sequences (x) by slicing the data
    x = torch.stack([data[i:i+block_size] for i in ix])
    # Create the target sequences (y) by slicing the data with a one-token offset
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
    return x, y

In [ ]:
# --- Test Cell for Exercise 1 ---
torch.manual_seed(42) # Set seed for reproducibility of the test

test_data = torch.arange(100)
test_batch_size = 4
test_block_size = 8

x, y = get_batch(test_data, test_block_size, test_batch_size)

# Test 1: Check output shapes
assert x.shape == (test_batch_size, test_block_size), f"Shape of x is wrong. Got {x.shape}, expected {(test_batch_size, test_block_size)}"
assert y.shape == (test_batch_size, test_block_size), f"Shape of y is wrong. Got {y.shape}, expected {(test_batch_size, test_block_size)}"
print("✅ Shape tests passed!")

# Test 2: Check the relationship between x and y
# For any sequence in the batch, y should be the "next token" for x.
# This means that y[:-1] should be equal to x[1:].
for i in range(test_batch_size):
    assert torch.equal(x[i, 1:], y[i, :-1]), f"Input x and target y are not correctly shifted at batch index {i}"
print("✅ Shift property test passed!")

# Test 3: Check that the values are correct based on the seed
expected_first_x_row = torch.tensor([81, 82, 83, 84, 85, 86, 87, 88])
assert torch.equal(x[0], expected_first_x_row), f"First row of x is incorrect. Got {x[0]}, expected {expected_first_x_row}"
expected_first_y_row = torch.tensor([82, 83, 84, 85, 86, 87, 88, 89])
assert torch.equal(y[0], expected_first_y_row), f"First row of y is incorrect. Got {y[0]}, expected {expected_first_y_row}"
print("✅ Value correctness test passed!")

print("\n🎉 All tests for get_batch passed! Great work.")

### Exercise 2: The Training Step

Awesome! Now that you can generate batches, let's implement the core logic of training: a single optimization step.

The `train_step` function will perform one forward pass and one backward pass for a given batch of data. Here are the four essential steps involved in any PyTorch training step:

1.  **Zero the gradients:** Before calculating new gradients, you must clear the old ones from the previous step. You can do this with `optimizer.zero_grad()`.
2.  **Forward pass:** Get the model's predictions (`logits`) and calculate the `loss` by passing the input batch (`xb`) and target batch (`yb`) to the model.
3.  **Backward pass:** Compute the gradients of the loss with respect to all model parameters by calling `loss.backward()`.
4.  **Update weights:** Tell the optimizer to update the model's parameters using the gradients computed in the backward pass. This is done with `optimizer.step()`.

Your job is to implement these four steps inside the `train_step` function.

In [ ]:
# GRADED FUNCTION: train_step

def train_step(model: nn.Module, optimizer: torch.optim.Optimizer, xb: torch.Tensor, yb: torch.Tensor) -> float:
    """
    Performs a single training step for the model.

    Args:
        model (nn.Module): The model to be trained.
        optimizer (torch.optim.Optimizer): The optimizer to update the model's parameters.
        xb (torch.Tensor): The input batch.
        yb (torch.Tensor): The target batch.

    Returns:
        float: The loss value for this step.
    """
        optimizer (torch.optim.Optimizer): The optimizer to update the model's parameters.
        xb (torch.Tensor): The input batch.
        yb (torch.Tensor): The target batch.

    Returns:
        float: The loss value for this step.
    """
    # 1. Set the gradients to zero
    optimizer.zero_grad(set_to_none=True)

    # 2. Perform a forward pass to get logits and loss
    logits, loss = model(xb, yb)

    # 3. Perform a backward pass to compute gradients
    loss.backward()

    # 4. Update the model's parameters
    optimizer.step()

    return loss.item()

    return loss.item()

In [ ]:
# --- Test Cell for Exercise 2 ---
torch.manual_seed(1337)

# Create a fresh model and optimizer for a clean test
test_model = DummyGPT()
test_optimizer = torch.optim.AdamW(test_model.parameters(), lr=learning_rate)

# Get a single batch of data
xb, yb = get_batch(train_data, block_size, batch_size)

# Store the value of a parameter before the training step
param_before = test_model.lm_head.weight.clone()

# Perform the training step
loss = train_step(test_model, test_optimizer, xb, yb)
param_after = test_model.lm_head.weight

# Test 1: Check if loss is a float
assert isinstance(loss, float), f"Loss should be a float, but got {type(loss)}"
print("✅ Loss type test passed!")

# Test 2: Check if parameters were updated
assert not torch.equal(param_before, param_after), "Model parameters were not updated after the training step!"
print("✅ Parameter update test passed!")

# Test 3: Check if gradients are zeroed after the step
# We perform another forward/backward pass to see if grads accumulate.
# If they were properly zeroed, the new grads should be independent of the old ones.
logits, loss2 = test_model(xb, yb)
loss2.backward()
grad1 = list(test_model.parameters())[0].grad.clone()

train_step(test_model, test_optimizer, xb, yb) # This step should zero the grads internally

logits, loss3 = test_model(xb, yb)
loss3.backward()
grad2 = list(test_model.parameters())[0].grad.clone()

assert not torch.allclose(grad1, grad2), "Gradients appear to be accumulating. Did you call optimizer.zero_grad()?"
print("✅ Gradient zeroing test passed!")


print("\n🎉 All tests for train_step passed! You're doing great.")

### Challenge Exercise (Optional): The Full Training Loop

You've built the two essential building blocks: getting data and training on it. Now, for the final challenge, let's put them together!

The `run_training` function orchestrates the entire process. It should loop for a specified number of steps (`num_steps`). In each step, it should:
1.  Get a new batch of data using your `get_batch` function.
2.  Perform a single training step using your `train_step` function.
3.  (For monitoring progress) Append the resulting loss to a list.

This is the rhythm of machine learning: get data, train, repeat.

In [ ]:
# GRADED FUNCTION: run_training (Optional)

def run_training(model: nn.Module, optimizer: torch.optim.Optimizer, data: torch.Tensor,
                 num_steps: int, batch_size: int, block_size: int) -> list[float]:
    """
    Runs the full training loop for a given number of steps.

    Args:
        model (nn.Module): The model to train.
        optimizer (torch.optim.Optimizer): The optimizer.
        data (torch.Tensor): The training data.
        num_steps (int): The number of training steps to perform.
        batch_size (int): The batch size.
        block_size (int): The context length.

    Returns:
        list[float]: A list of loss values, one for each training step.
    """
    losses = []
    model.train() # Set the model to training mode
        optimizer (torch.optim.Optimizer): The optimizer.
        data (torch.Tensor): The training data.
        num_steps (int): The number of training steps to perform.
        batch_size (int): The batch size.
        block_size (int): The context length.

    Returns:
        list[float]: A list of loss values, one for each training step.
    """
    losses = []
    model.train() # Set the model to training mode
    for step in range(num_steps):
        # 1. Get a batch of data
        xb, yb = get_batch(data, block_size, batch_size)

        # 2. Perform a training step and get the loss
        loss = train_step(model, optimizer, xb, yb)

        # 3. Store the loss
        losses.append(loss)
    return losses
    return losses

In [ ]:
# --- Integration Test ---
# This cell combines all your work. If everything is correct,
# you should see the model's loss decrease, which means it's learning!

print("Starting the full training run...")

# Re-initialize the model and optimizer to start fresh
torch.manual_seed(1337)
final_model = DummyGPT()
final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=learning_rate)

# Run the training loop
losses = run_training(final_model, final_optimizer, train_data, TRAINING_STEPS, batch_size, block_size)

# --- Final Assertions ---
# Test 1: Check if the number of losses recorded is correct
assert len(losses) == TRAINING_STEPS, f"Expected {TRAINING_STEPS} losses, but got {len(losses)}"
print("✅ Correct number of steps were run.")

# Test 2: Check if the model is learning (loss is decreasing)
# We take an average of the first 10 and last 10 losses.
# The final loss should be significantly lower than the initial one.
initial_loss = sum(losses[:10]) / 10
final_loss = sum(losses[-10:]) / 10

print(f"Initial average loss: {initial_loss:.4f}")
print(f"Final average loss:   {final_loss:.4f}")

assert final_loss < initial_loss, "The loss did not decrease! Something is wrong in the training loop."
print("✅ Loss has successfully decreased!")

print("\n\nCongratulations! You have successfully implemented a complete training loop!")
print("You have built the engine that powers modern AI models. Well done!")